# 🚗 License Plate Recognition — Adaptive Image Processing Pipeline

| Function | Member | Stage | Adaptive Behaviour |
|---|---|---|---|
| `m1_edge_detection` | M1 | Grayscale + CLAHE + Bilateral + Canny | Auto-tunes Canny thresholds via Otsu; CLAHE for dark/bright images |
| `m2_plate_candidate` | M2 | Morph Closing + Contour Filtering | Multi-pass search with relaxing constraints; scoring by position+area+aspect |
| `m3_perspective_binarize` | M3 | Perspective Correction + Binarization | Picks Otsu vs Adaptive based on local contrast; dynamic output size |


In [5]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets
import os, warnings
warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

Libraries loaded ✓


## 1: Adaptive Grayscale + Bilateral Filter + Edge Detection

In [6]:
def m1_edge_detection(image_path, visualize=True):
    original = cv2.imread(image_path)
    if original is None:
        raise FileNotFoundError(f"Cannot open image: {image_path}")

    h, w = original.shape[:2]
    gray_raw = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

    mean_intensity = np.mean(gray_raw)
    contrast = np.std(gray_raw)
    texture = cv2.Laplacian(gray_raw, cv2.CV_64F).var()

    # CLAHE: more conservative scaling (less noise amplification)
    clip = np.clip(2.0 + (50.0 / (contrast + 1e-5)), 2.0, 5.0)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    gray = clahe.apply(gray_raw)

    # Bilateral: driven more by texture than raw std
    base = max(5, min(15, int(min(h, w) / 60) * 2 + 1))
    sigma = np.clip(0.3 * contrast + 0.2 * texture**0.5, 10, 60)

    filtered = cv2.bilateralFilter(gray, base, sigma, sigma)

    # Canny: still Otsu-based but stabilized
    blurred = cv2.GaussianBlur(filtered, (5, 5), 0)
    otsu_val, _ = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    edge_density_bias = np.clip(texture / 1000.0, 0.8, 1.3)

    canny_low = int(np.clip(0.4 * otsu_val * edge_density_bias, 10, 120))
    canny_high = int(np.clip(1.2 * otsu_val * edge_density_bias, 80, 250))

    edges = cv2.Canny(filtered, canny_low, canny_high)

    meta = dict(
        clip=round(clip, 2),
        bilateral_d=base,
        sigma=round(float(sigma), 2),
        canny=[canny_low, canny_high],
        mean=int(mean_intensity),
        contrast=round(contrast, 1),
        texture=round(texture, 1)
    )

    if visualize:
        fig, axes = plt.subplots(1, 4, figsize=(20, 4))
        axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Original {w}x{h}")
        axes[1].imshow(gray_raw, cmap="gray")
        axes[1].set_title("Grayscale")
        axes[2].imshow(gray, cmap="gray")
        axes[2].set_title(f"CLAHE clip={clip:.2f}")
        axes[3].imshow(edges, cmap="gray")
        axes[3].set_title(f"Canny {canny_low}-{canny_high}")

        for ax in axes:
            ax.axis("off")

        plt.tight_layout()
        plt.show()

        print(meta)

    return original, gray, edges, meta

## 2: Adaptive Morph Closing + Contour Filtering

In [7]:
def m2_plate_candidate(original, edges, visualize=True):
    """
    M2: Morphology + contour ranking for license plate detection.

    Improvements over baseline:
      - Multi-kernel adaptive morphology (horizontal + square fallback)
      - Data-driven pass selection (no blind escalation)
      - Softer, more realistic scoring (fill, solidity, edge density)
      - Reduced hard-coded aspect anchoring
      - More stable confidence ranking

    Returns:
      plate_crop (BGR), best_contour, best_rect (x,y,w,h), meta
    """

    img_h, img_w = edges.shape
    img_area = img_h * img_w

    # adaptive kernel size
    kw = max(9, int(img_w * 0.022))
    if kw % 2 == 0:
        kw += 1

    # morphology kernels (orientation-aware)
    kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, 1))
    kernel_s = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, kw))

    morph_h = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel_h)
    morph_s = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel_s)
    dilated  = cv2.dilate(morph_h, np.ones((3, 3), np.uint8), iterations=1)

    candidates_src = [
        ("horizontal", morph_h),
        ("square", morph_s),
        ("dilated", dilated),
    ]

    best_contour = None
    best_rect = None
    best_score = -1
    best_source = None

    def score_candidate(cnt, x, y, w, h, src):
        area = w * h
        if area <= 0:
            return -1

        aspect = w / float(h + 1e-5)

        # soft aspect preference (not a hard gate)
        aspect_center = 4.0
        aspect_penalty = abs(aspect - aspect_center) / aspect_center
        aspect_score = max(0.0, 1.0 - aspect_penalty)

        # contour quality signals
        contour_area = cv2.contourArea(cnt)
        fill_ratio = contour_area / (area + 1e-5)
        solidity = contour_area / (cv2.arcLength(cnt, True) ** 2 + 1e-5)

        # edge support inside region
        region_edge = np.mean(edges[y:y+h, x:x+w]) / 255.0

        # position prior (soft, not binary)
        y_center = (y + h / 2) / img_h
        position_bias = 1.0 + 0.2 * (y_center > 0.3)

        # normalize area
        area_score = area / img_area

        # final score (balanced, not brittle)
        return (
            0.35 * fill_ratio +
            0.25 * solidity +
            0.20 * region_edge +
            0.15 * aspect_score +
            0.05 * area_score
        ) * position_bias

    # search across all morphology outputs (no sequential bias)
    for name, src in candidates_src:
        contours, _ = cv2.findContours(src, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)

            # quick reject tiny noise
            if w < 20 or h < 10:
                continue

            score = score_candidate(cnt, x, y, w, h, src)

            if score > best_score:
                best_score = score
                best_contour = cnt
                best_rect = (x, y, w, h)
                best_source = name

    # fallback if nothing meaningful found
    if best_contour is None:
        print("[M2] No valid plate candidate found — fallback to full image.")
        best_rect = (0, 0, img_w, img_h)
        best_contour = np.array([[[0,0]], [[img_w,0]], [[img_w,img_h]], [[0,img_h]]])
        best_source = "fallback"

    # padding (adaptive but safe)
    x, y, w, h = best_rect
    pad_x = max(4, int(w * 0.02))
    pad_y = max(4, int(h * 0.05))

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(img_w, x + w + pad_x)
    y2 = min(img_h, y + h + pad_y)

    best_rect = (x1, y1, x2 - x1, y2 - y1)
    plate_crop = original[y1:y2, x1:x2]

    meta = {
        "score": round(best_score, 4),
        "source": best_source,
        "kernel": kw,
        "rect": best_rect
    }

    if visualize:
        debug = original.copy()
        rx, ry, rw, rh = best_rect

        cv2.rectangle(debug, (rx, ry), (rx + rw, ry + rh), (0, 220, 60), 3)
        label = f"{best_source} | score={best_score:.3f}"
        cv2.putText(debug, label, (rx, max(ry - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 220, 60), 2)

        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        axes[0].imshow(morph_h, cmap="gray")
        axes[0].set_title("Morph (horizontal)")
        axes[1].imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
        axes[1].set_title("Best Detection")
        axes[2].imshow(cv2.cvtColor(plate_crop, cv2.COLOR_BGR2RGB))
        axes[2].set_title("Cropped Plate")

        for ax in axes:
            ax.axis("off")

        plt.tight_layout()
        plt.show()

        print("[M2]", meta)

    return plate_crop, best_contour, best_rect, meta

## 3: Adaptive Perspective Correction + Binarization

In [8]:
def m3_perspective_binarize(original, best_rect, edges=None, visualize=True):
    """
    M3: Perspective correction + adaptive binarization (robust version)

    Key upgrades:
      - Multi-candidate warping (quad + rotated rect + fallback)
      - Warp quality scoring instead of single-path trust
      - Realistic aspect ratio constraints (not strict inheritance from M2)
      - Data-driven binarization selection (Otsu vs Adaptive)
      - Post-warp validation signal

    Returns:
      binary_plate, best_warped, meta
    """

    gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
    x, y, w, h = best_rect

    crop_gray = gray[y:y+h, x:x+w]

    # ----------------------------
    # Step 1: edge + contour search
    # ----------------------------
    blurred = cv2.GaussianBlur(crop_gray, (3, 3), 0)
    _, crop_edges = cv2.threshold(
        blurred, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    contours, _ = cv2.findContours(
        crop_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # ----------------------------
    # Step 2: candidate geometry extraction
    # ----------------------------
    candidates = []

    # A) Best contour quad attempt
    if contours:
        biggest = max(contours, key=cv2.contourArea)
        peri = cv2.arcLength(biggest, True)

        for eps in [0.02, 0.03, 0.05, 0.08]:
            approx = cv2.approxPolyDP(biggest, eps * peri, True)
            if len(approx) == 4:
                quad = approx.reshape(4, 2).astype(np.float32)
                quad[:, 0] += x
                quad[:, 1] += y
                candidates.append(("quad", quad))
                break

    # B) Rotated rectangle fallback
    rect = cv2.minAreaRect(
        max(contours, key=cv2.contourArea)
    ) if contours else None

    if rect is not None:
        box = cv2.boxPoints(rect).astype(np.float32)
        box[:, 0] += x
        box[:, 1] += y
        candidates.append(("rotated_rect", box))

    # C) Hard fallback bounding box
    candidates.append((
        "bbox",
        np.array([[x, y], [x+w, y], [x+w, y+h], [x, y+h]], dtype=np.float32)
    ))

    # ----------------------------
    # Step 3: geometry ordering
    # ----------------------------
    def order_points(pts):
        rect = np.zeros((4, 2), dtype=np.float32)
        s = pts.sum(axis=1)
        d = np.diff(pts, axis=1).ravel()
        rect[0] = pts[np.argmin(s)]
        rect[2] = pts[np.argmax(s)]
        rect[1] = pts[np.argmin(d)]
        rect[3] = pts[np.argmax(d)]
        return rect

    # ----------------------------
    # Step 4: warp scoring function
    # ----------------------------
    def warp_score(warp):
        if warp is None:
            return -1

        g = cv2.cvtColor(warp, cv2.COLOR_BGR2GRAY)

        # character structure signal
        edge_var = cv2.Laplacian(g, cv2.CV_64F).var()

        # binarization stability proxy
        std = np.std(g)

        # foreground balance (not too empty, not too full)
        _, bin_tmp = cv2.threshold(
            g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        fg_ratio = np.mean(bin_tmp == 255)

        balance = 1.0 - abs(fg_ratio - 0.5)

        return (
            0.5 * min(edge_var / 500.0, 1.0) +
            0.3 * balance +
            0.2 * min(std / 80.0, 1.0)
        )

    # ----------------------------
    # Step 5: warp + evaluation
    # ----------------------------
    best_warp = None
    best_binary = None
    best_meta = None
    best_score = -1

    for mode, pts in candidates:
        src = order_points(pts)

        # adaptive output size (clamped realism)
        h_out = 80
        aspect = w / float(h + 1e-5)
        aspect = np.clip(aspect, 2.0, 6.5)
        w_out = int(np.clip(h_out * aspect, 160, 480))

        dst = np.array(
            [[0, 0], [w_out-1, 0], [w_out-1, h_out-1], [0, h_out-1]],
            dtype=np.float32
        )

        M = cv2.getPerspectiveTransform(src, dst)
        warped = cv2.warpPerspective(gray, M, (w_out, h_out))

        score = warp_score(warped)

        if score > best_score:
            best_score = score
            best_warp = warped
            best_meta = mode

    # ----------------------------
    # Step 6: adaptive binarization
    # ----------------------------
    local_std = float(np.std(best_warp))

    otsu_bin = cv2.threshold(
        best_warp, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]

    block = max(11, int(best_warp.shape[1] / 15) | 1)
    adaptive_bin = cv2.adaptiveThreshold(
        best_warp, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block, 4
    )

    def bin_quality(b):
        return np.mean(b == 255) * np.std(b)

    binary = otsu_bin if bin_quality(otsu_bin) > bin_quality(adaptive_bin) else adaptive_bin

    # fix polarity
    if np.mean(binary) < 127:
        binary = cv2.bitwise_not(binary)

    binary = cv2.medianBlur(binary, 3)

    meta = {
        "warp_mode": best_meta,
        "warp_score": round(best_score, 4),
        "local_std": round(local_std, 1),
        "out_size": best_warp.shape[::-1],
        "binarization": "otsu_vs_adaptive"
    }

    # ----------------------------
    # Step 7: visualization
    # ----------------------------
    if visualize:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))

        axes[0].imshow(gray[y:y+h, x:x+w], cmap="gray")
        axes[0].set_title("Input Crop (M2)")

        axes[1].imshow(best_warp, cmap="gray")
        axes[1].set_title(f"Warped [{best_meta}] score={best_score:.3f}")

        axes[2].imshow(binary, cmap="gray")
        axes[2].set_title(f"Binary std={local_std:.0f}")

        for ax in axes:
            ax.axis("off")

        plt.tight_layout()
        plt.show()

        print("[M3]", meta)

    return binary, best_warp, meta

## Combined Pipeline

In [9]:
def run_pipeline(image_path, visualize=True):
    """Run M1 -> M2 -> M3 with lightweight validation and failure awareness."""

    sep = "─" * 55

    def log(stage, msg):
        print(sep)
        print(f"  {stage}  {msg}")
        print(sep)

    print(sep)
    print("  PIPELINE START")
    print(sep)

    # -----------------------
    # M1
    # -----------------------
    log("M1", "Grayscale + CLAHE + Bilateral + Auto-Canny")

    original, gray, edges, m1 = m1_edge_detection(
        image_path, visualize=visualize
    )

    m1_quality = np.mean(edges) / 255.0

    # crude sanity check
    if m1_quality < 0.01:
        print("⚠️  M1 warning: extremely low edge signal")

    # -----------------------
    # M2
    # -----------------------
    log("M2", "Morph Closing + Multi-pass Contour Filter")

    plate_crop, contour, best_rect, m2 = m2_plate_candidate(
        original, edges, visualize=visualize
    )

    m2_score = m2.get("score", 0)

    if m2_score < 0.002:
        print("⚠️  M2 warning: weak plate confidence")

    # fallback detection
    x, y, w, h = best_rect
    if w == original.shape[1] and h == original.shape[0]:
        print("⚠️  M2 fallback: full-image crop used")

    # -----------------------
    # M3
    # -----------------------
    log("M3", "Perspective Warp + Adaptive Binarization")

    binary, warped, m3 = m3_perspective_binarize(
        original, best_rect, visualize=visualize
    )

    m3_score = m3.get("warp_score", 0)

    if m3_score < 0.2:
        print("⚠️  M3 warning: low warp confidence (geometry unstable)")

    # -----------------------
    # summary
    # -----------------------
    all_meta = {"M1": m1, "M2": m2, "M3": m3}

    print("\n✅ PIPELINE COMPLETE\n")

    for k, v in all_meta.items():
        print(f"[{k}] {v}")

    # optional global sanity flag
    global_confidence = (m2_score + m3_score) / 2

    print(f"\n Global confidence: {global_confidence:.3f}")

    if global_confidence < 0.15:
        print(" Overall result is unstable — expect OCR failure")

    return binary, all_meta

## Interactive Widget — Upload & Run

In [10]:
upload  = widgets.FileUpload(accept='image/*', multiple=False, description='Upload')
run_btn = widgets.Button(description='Run Pipeline', button_style='success')
out     = widgets.Output()

def _on_run(b):
    out.clear_output(wait=True)
    with out:
        if not upload.value:
            print('Please upload an image first.'); return
        fname = list(upload.value.keys())[0]
        data  = upload.value[fname]['content']
        path  = f'/tmp/{fname}'
        with open(path, 'wb') as f: f.write(bytes(data))
        try:
            run_pipeline(path, visualize=True)
        except Exception as e:
            print(f'Error: {e}')

run_btn.on_click(_on_run)
display(widgets.HBox([upload, run_btn]), out)

Output()

## Quick Test — Run on a Local File Path

In [11]:
IMAGE_PATH = 'car.jpg'   # <-- change to your image

if os.path.exists(IMAGE_PATH):
    binary_result, meta = run_pipeline(IMAGE_PATH, visualize=True)
else:
    print(f'File not found: {IMAGE_PATH}')
    print('Set IMAGE_PATH to a valid image in this folder.')

File not found: car.jpg
Set IMAGE_PATH to a valid image in this folder.
